# iot-sae-ids — Colab runner

Runs the IoT sparse-autoencoder IDS experiments on Colab Pro (T4 GPU). Runtime → Change runtime type
→ **T4 GPU** (High-RAM recommended: the exact RBF SVM on UNSW-NB15 uses ~96k training rows).

**Before the first run**, put the raw data on Google Drive under `WORK_DIR` (default `MyDrive/iot-sae-ids-work`):

```
iot-sae-ids-work/data/raw/
├── ton_iot/Train_Test_Network.csv
├── unsw_nb15/UNSW_NB15_training-set.csv
├── unsw_nb15/UNSW_NB15_testing-set.csv
└── rt_iot2022/RT_IOT2022.csv      # optional: fetched from UCI if missing
```

Data, cached features, tuning trials and **all results live on Drive** (`IDS_DATA_ROOT`,
`IDS_RESULTS_ROOT`). After a disconnect: rerun sections 0–2, then rerun the step you were on —
finished tuning trials and finished runs are skipped.

| Step | What | Rough T4 time |
|---|---|---|
| 3 | tests | ~2 min |
| 4 | smoke test (2% sample, 12 runs) | ~15 min |
| 5 | SAE tuning, 20 random trials × 3 datasets | ~1–3 h |
| 6a | grid: RF + SVM + SAE, 5 seeds | several hours (exact SVM on UNSW-NB15 dominates) |
| 6b | grid: LSTM, 5 seeds | several hours |

## 0. Settings

In [ ]:
WORK_DIR    = "/content/drive/MyDrive/iot-sae-ids-work"
REPO_SOURCE = "github"   # "github" or "bundle" (git bundle at WORK_DIR/iot-sae-ids.bundle)
REPO_URL    = "https://github.com/esmailga/iot-sae-ids.git"
BRANCH      = "main"
CODE_DIR    = "/content/iot-sae-ids"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
pathlib.Path(WORK_DIR, "data", "raw").mkdir(parents=True, exist_ok=True)
pathlib.Path(WORK_DIR, "results").mkdir(parents=True, exist_ok=True)

## 1. Get the code (clone, or update to the latest commit)

In [ ]:
import subprocess
src = os.path.join(WORK_DIR, "iot-sae-ids.bundle") if REPO_SOURCE == "bundle" else REPO_URL
if not os.path.isdir(os.path.join(CODE_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, src, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", src, f"{BRANCH}:refs/remotes/src/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "reset", "--hard", f"src/{BRANCH}"], check=True)
!git -C {CODE_DIR} log --oneline -3

## 2. Install and point paths at Drive

In [ ]:
%cd {CODE_DIR}
!pip install -q -r requirements-tf.txt && pip install -q -e .
os.environ["IDS_DATA_ROOT"] = os.path.join(WORK_DIR, "data")
os.environ["IDS_RESULTS_ROOT"] = os.path.join(WORK_DIR, "results")
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Tests (TensorFlow tests run here)

In [ ]:
!python -m pytest -q

## 4. Smoke test — all 12 combinations on a 2% sample
Writes `results/tables/smoke_test.md` on Drive.

In [ ]:
!python scripts/smoke_test.py --sample 0.02

## 5. SAE hyperparameter tuning (before the grid)

Random search (20 trials) per dataset on a 20% stratified subsample of the train split; selection by
validation ROC-AUC. Writes `results/best_sae_<dataset>.yaml`, which the grid uses automatically.
Resumable trial-by-trial. For the full 216-combination grid add `--mode grid`.

In [ ]:
!python scripts/tune_sae.py --dataset ton_iot unsw_nb15 rt_iot2022

In [ ]:
!for f in {WORK_DIR}/results/best_sae_*.yaml; do echo "== $f"; cat "$f"; done

## 6. Full grid — 4 models × 3 datasets × 5 seeds (resumable)

Check what is left with the dry run. Each cell can be re-run after a disconnect; finished runs
are skipped and failures go to `results/raw/failures.log` without stopping the grid.

In [ ]:
!python scripts/run_all.py --dry-run 2>&1 | grep -E 'Grid|WARNING'

### 6a. RF, SVM and SAE

In [ ]:
!python scripts/run_all.py --models sae rf svm

### 6b. LSTM (slowest; recurrent dropout disables cuDNN)

In [ ]:
!python scripts/run_all.py --models lstm

## 7. Progress and failures

In [ ]:
import glob, json, collections
runs = [p for p in glob.glob(f"{WORK_DIR}/results/raw/*.json") if "__frac" not in p]
c = collections.Counter(json.load(open(p))["run"]["model"] for p in runs)
print(f"{len(runs)}/60 full runs done:", dict(c))
fl = f"{WORK_DIR}/results/raw/failures.log"
print(open(fl).read()[-3000:] if os.path.exists(fl) else "no failures")

## 8. Package results for the local repo

Downloads a zip of the per-run JSONs, score files, tuned SAE configs and tables. Unzip it into the
repo root locally; `results/raw/` is git-ignored, tables/figures are generated from it in Phase 7.

In [ ]:
!cd {WORK_DIR} && rm -f results_bundle.zip && zip -qr results_bundle.zip results/raw results/best_sae_*.yaml results/tables results/features -x '*__frac*'
from google.colab import files
files.download(f"{WORK_DIR}/results_bundle.zip")